<a href="https://colab.research.google.com/github/p-tech/ntu-growth-hacking/blob/main/clustering/Notebook_6_Customer_Clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 6 · Which customers order in similar ways?
**NTU MSc Growth Hacking | Segmentation and targeting**

Oddbox wants to test a follow-up message. Would you send the same message to a household that ordered once and one that ordered repeatedly? Work with eight fictional customer records before tackling the 1,000-customer file in Notebook 7.

Open in Colab, save a copy and run the cells in order. Pause at each question before revealing the next result. You will group customers yourself, follow one K-means calculation and decide whether an extra group would change a marketing decision.

## 1 · Draw your own groups
Each row represents one household over the same 12 months. The two features (measurements) are fruit and veg box orders and fruit booster orders. Zero means no recorded orders of that product. A blank would mean the count was unknown. A repeated customer row would give that household extra influence.

Run the cell and sketch a boundary around two groups. Write down the customer letters in each group. Compare with a neighbour: which customer was hardest to place? Why?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from IPython.display import display

customers = pd.DataFrame({
    'customer': list('ABCDEFGH'),
    'box_orders': [1,2,2,5,8,9,9,10],
    'booster_orders': [0,0,1,1,1,1,5,6]
}).set_index('customer')
assert customers.index.is_unique and customers.notna().all().all()
points = customers.to_numpy(dtype=float)
display(customers)

def plot_groups(title, groups=None, centres=None):
    fig, ax = plt.subplots(figsize=(8,4.8))
    palette = ['#2166ac', '#b35806', '#188264']
    for j, (name, (x,y)) in enumerate(zip(customers.index, points)):
        ax.scatter(x,y,color=palette[groups[j]] if groups is not None else '#555555',s=70)
        ax.annotate(name,(x,y),xytext=(5,6),textcoords='offset points')
    if centres is not None:
        for j,(x,y) in enumerate(centres):
            ax.scatter(x,y,marker='X',s=160,color=palette[j],edgecolor='black',label=f'Centre {j+1}')
        ax.legend()
    ax.set(xlabel='Fruit and veg box orders',ylabel='Fruit booster orders',
           xlim=(0,11),ylim=(-1,7),title=title)
    ax.set_aspect('equal')
    ax.grid(alpha=.2)
    plt.show()

plot_groups('Which households would you group together?')

## 2 · Assign each customer to a starting centre
In **K-means**, *K* is the number of groups you ask the computer to find. It starts with a centre for each group and assigns each customer to the nearest centre. This is **unsupervised learning**: we supply the order counts and the number of groups. There are no previous segment labels to learn from.

For this demonstration, start two centres at A (1, 0) and D (5, 1). Look at B (2, 0). It is one order away from A. To reach D, it is three box orders and one booster order away. Its straight-line distance is √(3² + 1²), about 3.16. Predict B's group, then check the table. Which centre initially attracts E?

In [ ]:
start = customers.loc[['A','D']].to_numpy(dtype=float)
def distances_to(centres):
    return np.linalg.norm(points[:,None,:] - centres[None,:,:], axis=2)

distance = distances_to(start)
first = distance.argmin(axis=1)
display(pd.DataFrame({'distance to A':distance[:,0], 'distance to D':distance[:,1],
                      'first group':first+1},index=customers.index).round(2))
plot_groups('First assignment using A and D as centres',first,start)

## 3 · Move the centres. Does anyone switch?
A centre now moves to the **mean** of its members. For the group containing A, B and C, average the box counts (1, 2, 2) and the booster counts (0, 0, 1) separately. Its new centre is about (1.67, 0.33). A centre can lie between real households.

The other centre moves too. Watch customer D: it began at a centre of its own. Is that still the nearest group once the frequent buyers pull the centre towards them? Run the cell. Each row shows D’s actual group at one stage, alongside the other customers.

In [ ]:
groups = first.copy()
history = [groups.copy()]
for step in range(1, 20):
    centres = np.stack([points[groups == g].mean(axis=0) for g in range(2)])
    updated = distances_to(centres).argmin(axis=1)
    history.append(updated.copy())
    print(f'Update {step}: centres {centres.round(2).tolist()}; switched: {list(customers.index[updated != groups])}')
    if np.array_equal(updated, groups):
        break
    groups = updated
display(pd.DataFrame(np.array(history)+1, columns=customers.index,
                     index=['Starting groups']+[f'After update {i}' for i in range(1,len(history))]))
plot_groups('Final groups once assignments stop changing',groups,centres)

## 4 · Would a third group change your decision?
The walkthrough used fixed starting points so everyone could follow the arithmetic. Here K-means tries several starts for each K and keeps the result with the lowest total squared distance to the centres. The group numbers are just labels.

Compare K = 2 and K = 3. The **silhouette score** asks whether customers sit closer to members of their own group than to members of the nearest other group. Values near 1 indicate clearer separation, near 0 indicate a boundary, and negative values suggest a customer may be closer on average to another group. The table shows the average across all eight households.

Before running the cell, predict which customers could form a third group. Afterwards, compare its size, average box orders and average booster orders. Would that group receive a different offer?

In [ ]:
solutions = {}
for k in (2,3):
    fit = KMeans(n_clusters=k, n_init=20, random_state=42).fit(points)
    solutions[k] = fit
    members = customers.assign(group=fit.labels_+1)
    print(f'K = {k} | mean silhouette = {silhouette_score(points,fit.labels_):.2f}')
    display(members.groupby('group').agg(households=('box_orders','size'),
            mean_box_orders=('box_orders','mean'),mean_booster_orders=('booster_orders','mean')).round(2))
    plot_groups(f'K = {k}: which behaviour distinguishes each group?',fit.labels_,fit.cluster_centers_)

## 5 · Turn one group into a test
Select your preferred K below. Give one group a short name based on its observed orders. “Frequent box buyers who rarely add boosters” is supported by purchase counts. Calling them “price-sensitive families” would need more evidence. In particular, one order could reflect a customer joining late in the year.

**Your recommendation:** We chose K = ___ because ___. Group ___ contains ___ households. Their orders suggest ___. We would test ___ against ___ and measure ___.

For example, you could test a fruit booster suggestion against the usual follow-up among frequent box buyers who rarely add boosters. Measure the proportion adding a booster to their next order. Eight teaching records are enough to practise the reasoning. A live test would need a larger eligible group.

In [ ]:
chosen_k = 2 #@param [2,3] {type:"raw"}
result = customers.assign(group=solutions[chosen_k].labels_+1, k=chosen_k)
display(result.sort_values('group'))
# Optional download. Tick the box and rerun this cell.
download_csv = False #@param {type:"boolean"}
if download_csv:
    from google.colab import files
    filename = 'Notebook-6-customer-groups.csv'
    result.to_csv(filename)
    files.download(filename)

**Before Notebook 7:** We gave a difference of one box order the same weight as a difference of one booster order. What would happen to the map if you added household size or income band? Decide which features should shape the groups and which might be more useful for describing them afterwards.